# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring the [FAIR\^2 mass spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and explore the available records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
meta = dataset.metadata
print(f"Dataset Name: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Authors: {meta.author}")
print(f"Published: {meta.datePublished}")
print(f"License: {meta.license}")

## 2. Data Overview
List available `RecordSet` objects and summarize their fields. 

All entities are referenced by their unique `@id` fields, as per Croissant best practice.

In [ ]:
# List all RecordSet @ids and summarize their fields
record_sets = dataset.metadata.record_sets
print("Available RecordSets and their fields:")

for rs in record_sets:
    print(f"- RecordSet: @id={rs['@id']}, name={rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for fld in fields:
        if isinstance(fld, dict):
            print(f"    - Field: @id={fld['@id']}, name={fld.get('name', 'N/A')}, dataType={fld.get('dataType', 'N/A')}")
        else:
            print(f"    - Field: @id={fld}, (no details loaded)")
    print()

## 3. Data Extraction
Load data from a specific RecordSet into a DataFrame for analysis.

Specify the RecordSet and field `@id`s from the overview above.

In [ ]:
# For illustration, select the @id of the main RecordSet for protein abundance (edit as appropriate for your dataset)
main_record_set_id = None
for rs in dataset.metadata.record_sets:
    if "protein" in rs.get('name', '').lower() or "abundance" in rs.get('name', '').lower():
        main_record_set_id = rs['@id']
        break

# If no match found, just pick the first RecordSet
if not main_record_set_id and dataset.metadata.record_sets:
    main_record_set_id = dataset.metadata.record_sets[0]['@id']

print(f"Selected RecordSet ID for analysis: {main_record_set_id}")

# List all record set ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found in RecordSet {record_set_id}.")

if main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print(f"Columns in RecordSet {main_record_set_id}:\n{df.columns.tolist()}")
    display(df.head())
else:
    print("Selected RecordSet did not load any data.")

## 4. Exploratory Data Analysis (EDA)
Process and analyze numeric fields such as protein abundance measures.

We'll select a numeric field by its `@id` (from the overview), filter, normalize, and group as appropriate for the dataset.

In [ ]:
# Select an example numeric field. Replace with the actual @id if known.
import numpy as np

if main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Try to pick a numeric column automatically that might represent abundance/amount/peptide count
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float32, np.float64, np.int64, np.int32, float, int] or 'count' in col.lower() or 'abund' in col.lower() or 'mw' in col.lower()]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        # Remove outliers (e.g. protein abundance > 100)
        threshold = df[numeric_field].quantile(0.95)
        filtered_df = df[df[numeric_field] < threshold]
        print(f"Filtered records with {numeric_field} below 95th percentile ({threshold:.2f}): {len(filtered_df)} records")

        # Normalize
        filtered_df[f'{numeric_field}_zscore'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f'{numeric_field}_zscore']].head())

        # Try grouping by a categorical field, e.g. 'description' if available
        group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
            print(f"Grouped by {group_field} (top 5):")
            print(grouped_df.head())
    else:
        print("No obvious numeric field found to analyze.")
else:
    print("Main RecordSet data unavailable.")

## 5. Visualization
Visualize data distributions for quantitative fields.

We'll plot a histogram and boxplot for the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id in dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f'Histogram of {numeric_field}')

    plt.subplot(1, 2, 2)
    sns.boxplot(x=filtered_df[numeric_field])
    plt.title(f'Boxplot of {numeric_field}')
    plt.show()

    # If the grouping field was found, plot the top 10 groups
    if 'grouped_df' in locals() and group_field:
        top = grouped_df.head(10)
        plt.figure(figsize=(10, 5))
        sns.barplot(y=top[group_field], x=top[numeric_field], orient='h')
        plt.title(f'{numeric_field} by {group_field} (top 10 groups)')
        plt.show()

## 6. Conclusion
We have loaded and explored the FAIR\^2 dataset using its Croissant schema and the `mlcroissant` library.

- We reviewed available record sets and fields (using `@id` per Croissant best practice),
- Extracted data for further analysis and performed basic normalization and grouping,
- Visualized key numeric distributions.

For advanced analyses—such as detailed protein identification, cross-record set joins, or longitudinal studies—consult the field documentation and schema links in the metadata. Use `@id` references when linking or joining entities across the dataset.

For more, see the [mlcroissant documentation](https://mlcroissant.ai/).